# Tutorial: Service Principal Auth for Automation Comparison

Audience:
- presenters who want to explain app-only Power BI access
- engineers comparing user-context delivery and automation-oriented auth models

Prerequisites:
- `.env` filled in locally, including `CLIENT_SECRET`
- service principal tenant settings completed in `docs/operations/local-setup.md`
- service principal added to the target workspace if required

Learning goals:
- acquire an app-only token with client credentials flow
- run the same metadata calls as the delegated notebook where supported
- explain where service principal fits in automation and where it does not
- keep service principal as an optional comparison path, not the default demo


## Outline

1. Load config
2. Acquire a service principal token
3. List workspaces
4. List datasets and reports
5. Optionally attempt a DAX query

Use this notebook only after the delegated notebook is already working. Treat it as the automation comparison path, especially for supported non-RLS and non-SSO scenarios.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.auth.service_principal import acquire_service_principal_access_token
from src.powerbi.client import PowerBIClient
from src.config.loader import load_config, require_dataset_id, require_group_id

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)


## Step 1 - Load config

This shows a redacted summary. The secret is never printed.

In [ ]:
config = load_config()
config.redacted_summary()


## Step 2 - Acquire a service principal token

If this fails, do not try to fix it in code first. Check tenant settings, allowed security groups, workspace membership, and the local client secret.

In [ ]:
access_token = acquire_service_principal_access_token(config)
client = PowerBIClient(access_token, timeout_seconds=config.request_timeout_seconds)
print("Service principal access token acquired.")


## Step 3 - List workspaces

If the target workspace does not appear here, the service principal usually lacks workspace membership or is blocked by tenant settings.

In [ ]:
workspaces_df = pd.DataFrame(client.get_groups())
workspaces_df


## Step 4 - List datasets and reports

These are useful metadata calls for an automation story. They are usually safer to demo than `executeQueries` under service principal auth.

In [ ]:
group_id = require_group_id(None, config)
datasets_df = pd.DataFrame(client.get_datasets_in_group(group_id))
reports_df = pd.DataFrame(client.get_reports_in_group(group_id))

print("Datasets")
display(datasets_df)
print("Reports")
display(reports_df)


## Step 5 - Optional DAX query

Use this only if you already know the dataset does not hit unsupported service principal identity limitations such as RLS or SSO for `executeQueries`.

In [ ]:
dataset_id = require_dataset_id(None, config)
print("If this step fails, the limitation may be by design for the dataset or tenant configuration.")
query_df = pd.DataFrame(
    client.execute_queries_in_group(
        group_id=group_id,
        dataset_id=dataset_id,
        dax_query=config.dax_query,
        impersonated_user_name=config.impersonated_user_name,
    )
)
query_df


## Presenter notes

- This path is optional and admin-dependent.
- It is useful for automation narratives, not as the simplest trial demo.
- If this breaks while delegated auth works, that difference is part of the lesson.
